# Libya Automated Land-Cover Mapping — Python / Earth Engine

A companion notebook for the personal portfolio implementation by **Hamed Sabzchi Dehkharghani**. It first validates the public Earth Engine inputs, then provides optional cells for the private AOI/training assets and the full Random Forest workflow.


In [ ]:
!pip -q install earthengine-api geemap

import ee
PROJECT_ID = 'practical-proxy-441422-n6'
ee.Authenticate(auth_mode='notebook', force=True)
ee.Initialize(project=PROJECT_ID)
print('SUCCESS: Earth Engine initialized.')


In [ ]:
import importlib.util, pathlib, sys, urllib.request

ENGINE_URL = 'https://raw.githubusercontent.com/hamedsabzchi/libya-automated-land-cover-mapping/main/python/landcover_engine.py'
engine_path = pathlib.Path('landcover_engine.py')
urllib.request.urlretrieve(ENGINE_URL, engine_path)
spec = importlib.util.spec_from_file_location('landcover_engine', engine_path)
lc = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lc
spec.loader.exec_module(lc)
print('SUCCESS: land-cover engine loaded from GitHub.')


## Quick public-data validation

This cell does **not** need the personal training assets. It checks the current 2024 Satellite Embedding layer over Libya and displays three embedding axes for a lightweight visual smoke test.


In [ ]:
import geemap

check = lc.public_input_smoke_test(2024)
print(check)
assert check['image_count'] > 0
assert check['band_count'] == 64
print('SUCCESS: public Satellite Embedding input is available.')

libya = lc.libya_boundary()
embedding = lc.get_embedding(2024, libya.geometry())
m = geemap.Map(center=[27.0, 17.0], zoom=5)
m.addLayer(embedding, {'bands':['A01','A16','A09'], 'min':-0.3, 'max':0.3}, '2024 Satellite Embedding')
m.addLayer(libya.style(color='111111', fillColor='00000000', width=2), {}, 'Libya boundary')
m


## Check the default personal AOI and training assets

Run this only from an Earth Engine account that has access to the default assets. The assets are external dependencies and are not stored in GitHub.


In [ ]:
config = lc.LandCoverConfig()
lc.validate_config(config)
roi, points = lc.load_assets(config)
print(lc.configuration_summary(config))
print('AOI features:', roi.size().getInfo())
print('Valid training points:', points.size().getInfo())
print('SUCCESS: default personal assets are accessible.')


## Optional full model run — one seed

This is computationally heavier. It trains one Random Forest per selected year, applies the temporal decision rules, and evaluates the hold-out sample. The reference JavaScript app can test multiple seeds; use `lc.search_seeds(config)` only when you intentionally want the full seed search.


In [ ]:
model = lc.build_seed_model(config, seed=42)
metrics = lc.metrics_dictionary(model).getInfo()
print('Overall accuracy:', metrics['overall'])
print('Kappa:', metrics['kappa'])

landcover_map = geemap.Map(center=[27.0, 17.0], zoom=6)
landcover_map.addLayer(model['final_map'], {'min':1, 'max':12, 'palette':list(lc.CLASS_PALETTE)}, 'Land-cover classification')
landcover_map.addLayer(model['roi'].style(color='111111', fillColor='00000000', width=2), {}, 'AOI')
landcover_map


## Interpretation

The accuracy reported above is an internal class-stratified random hold-out assessment from the supplied training asset. It is not independent external validation. The final multi-year map also applies explicit priority rules for cultivated rainfed (class 4) and cultivated irrigated (class 5), so it should be interpreted as the output of the documented supervised + temporal-rule workflow.
